# Crime Trends 

## Introduction

This project demonstrates comprehensive crime pattern analysis and predictive modeling using machine learning techniques applied to Los Angeles Police Department crime data from 2020 to present. Leveraging over one million crime incidents, the analysis performs automated crime type classification and geographic trend analysis to support law enforcement decision-making and public safety initiatives.

**Key Features:**
- Multi-algorithm classification with Decision Trees, Random Forest, and Logistic Regression
- Advanced data preprocessing with label encoding and feature engineering
- Interactive geographic visualization using Folium heat maps
- Comprehensive exploratory data analysis with temporal and demographic insights
- Real-time crime type prediction based on incident characteristics and location data

**Technical Highlights:**
- **Dataset**: 1M+ crime incidents with 28 features including geographic coordinates, victim demographics, and case details
- **Machine Learning Pipeline**: Feature scaling, train-test splitting, and cross-validation for robust model evaluation
- **Classification Task**: Multi-class prediction of crime types (vehicle theft, burglary, robbery, etc.) using incident characteristics
- **Visualization**: Interactive crime heat maps and statistical trend analysis for geographic hotspot identification
- **Performance Metrics**: Accuracy assessment across multiple algorithms to identify optimal classification approach

**Practical Applications:**
- Law enforcement resource allocation and patrol optimization
- Crime hotspot identification for community safety initiatives
- Automated incident classification for police reporting systems
- Predictive analytics for proactive crime prevention strategies

This analysis transforms raw crime data into actionable intelligence, enabling data-driven approaches to urban safety and law enforcement operations while demonstrating the power of machine learning in public safety applications.



To enhance the functionality of the CoreAI  environment, we need to install some libraries not pre-installed but required for this notebook. 

## Pre-requisites
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment:

```bash
export PROJECT_NAME="Crime-Trends"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}-myvenv --display-name="Python (${PROJECT_NAME}-myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}-myvenv)"
```

## Install Required Libraries:

Before running the following command in jupyter notebook, make sure you are in the directory where the Jupyter Notebook and virtual environment is located. This ensures the ./ path is always current. You can use the cd command to change to your project directory and pwd to verify your current directory.


In [ ]:
import os
def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)

set_env_with_cache_dir("PIP_CACHE_DIR", "pip")
set_env_with_cache_dir('KAGGLEHUB_CACHE', 'data')


In [ ]:
!. ./myvenv/bin/activate; pip install -r requirements.txt

## Download dataset (if not already downloaded)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sonawanelalitsunil/crime-trends-2020present")

print("Path to dataset files:", path)

## Importing Libraries
Imports essential libraries for data manipulation (pandas, numpy), visualization (seaborn, matplotlib), and machine learning models for crime data analysis.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay

# ML Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


## Dataset Loading and Path Verification
Loads the crime dataset from the cached Kaggle download location and verifies the file path exists before reading the CSV data. Also, you can change the path of the CSV file to explore the dataset manually.

In [ ]:
csv_path = files_dir = os.path.join(os.environ['KAGGLEHUB_CACHE'], "datasets/sonawanelalitsunil/crime-trends-2020present/versions/1/Crime_Data_from_2020_to_Present.csv")
if not os.path.exists(csv_path):
    print(f"!!!!! Path ({DIR_PATH} not found -- Must fix")
df = pd.read_csv(csv_path) # Use this if your data is in a CSV file


## Dataset Exploration
Displays the first and last 5 rows of the dataset to understand data structure and recent entries.
Provides statistical summary of numerical columns to identify data ranges, distributions, and potential outliers.


In [ ]:
df.head()  # Shows first 5 rows
df.tail()  # Shows last 5 rows
df.describe()  # Shows statistical summary

## Dataset Dimensions and Column Structure
Lists all column names to provide a complete overview of available data fields.


In [ ]:
df.columns

## Dataset Information and Missing Values Analysis
Displays basic dataset information including data types, non-null counts, and memory usage.
Identifies missing values across all columns to understand data completeness and quality issues.


In [ ]:
df.info()
df.isnull().sum()

## Duplicate Record Detection and Analysis
Identifies and counts duplicate records in the dataset based on key identifiers.
Ensures data integrity by highlighting potential data entry errors or system duplicates.


In [ ]:
df.duplicated().sum()

## Missing Data Pattern Visualization
Identifies columns with significant missing values that may require special handling.

In [ ]:
df.isnull().sum().sum()
df.shape

## Data Visualizations

## Monthly Crime Trend Analysis
Converts the date column to datetime format and groups crime incidents by month to analyze temporal patterns.
Creates a line plot showing the monthly trend of crime occurrences over time to identify seasonal variations and long-term changes.


In [ ]:
# Group by month and count
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'])
monthly_trend = df.groupby(df['DATE OCC'].dt.to_period('M')).size()

monthly_trend.plot(kind='line', figsize=(12, 5), title='Monthly Crime Trend')

## Hourly Crime Distribution Analysis
Extracts hour information from the time occurrence data and analyzes crime frequency patterns throughout the day.
Creates a bar chart showing crime counts by hour to identify peak crime periods and daily temporal patterns for law enforcement planning.


In [ ]:
df['HOUR'] = df['TIME OCC'] // 100
df['HOUR'].value_counts().sort_index().plot(kind='bar', title='Crimes by Hour')

## Crime Distribution by Police Areas
Analyzes crime frequency across different LAPD patrol areas to identify high-crime neighborhoods and resource allocation needs.
Creates a horizontal bar chart showing crime counts by area name, making it easy to compare which districts experience the most criminal activity.


In [ ]:
df['AREA NAME'].value_counts().plot(kind='barh', title='Crimes by Area')

## Top 10 Most Common Crime Types Analysis
Identifies and visualizes the most frequently occurring crime types in Los Angeles from the dataset.
Creates a bar chart showing the top 10 crime categories to understand which offenses dominate criminal activity and require priority attention.


In [ ]:
df['Crm Cd Desc'].value_counts().head(10).plot(kind='bar', title='Top 10 Crime Types')

## Victim Sex Distribution Analysis
Analyzes the gender composition of crime victims using a pie chart to show proportional representation.
Provides visual insight into gender-based victimization patterns to inform targeted safety programs and resource allocation.


In [ ]:
df['Vict Sex'].value_counts().plot(kind='pie', autopct='%1.1f%%', title='Victim Sex Distribution')

## Top 10 Victim Descent Groups Analysis
Examines the demographic composition of crime victims by ethnic/racial background to identify disproportionate impacts.
Creates a bar chart showing the most affected descent groups to guide culturally sensitive policing and community outreach initiatives.


In [ ]:
df['Vict Descent'].value_counts().head(10).plot(kind='bar', title='Top 10 Victim Descent Groups')

## Top Weapons Used in Crimes Analysis
Identifies the most frequently used weapons in criminal incidents to understand threat patterns and violence levels.
Visualizes the top 10 weapon types through a bar chart to inform law enforcement training and public safety strategies.


In [ ]:
df['Weapon Desc'].value_counts().head(10).plot(kind='bar', title='Top Weapons Used')

## Interactive Crime Heat Map Generation
Creates an interactive Folium heat map showing crime density across Los Angeles using geographic coordinates from the dataset.
Processes latitude/longitude data to generate a web-based visualization highlighting crime hotspots and spatial patterns for geographic analysis.


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# 1. Load your dataset
csv_path = files_dir = os.path.join(os.environ['KAGGLEHUB_CACHE'], "datasets/sonawanelalitsunil/crime-trends-2020present/versions/1/Crime_Data_from_2020_to_Present.csv")
if not os.path.exists(csv_path):
    print(f"!!!!! Path ({DIR_PATH} not found -- Must fix")
df = pd.read_csv(csv_path)  # Use this if your data is in a CSV file

# 2. Ensure columns are numeric
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')

# 3. Remove rows with missing or out-of-range coordinates
df_clean = df.dropna(subset=['LAT', 'LON'])
df_clean = df_clean[
    (df_clean['LAT'] >= -90) & (df_clean['LAT'] <= 90) &
    (df_clean['LON'] >= -180) & (df_clean['LON'] <= 180)
]

# 4. Remove duplicated coordinate pairs (optional, but good for visualization)
df_clean = df_clean.drop_duplicates(subset=['LAT', 'LON'])

# 5. Check that data remains
if df_clean.empty:
    print("No valid data points for the heat map!")
else:
    # 6. Create the map centered on the cleaned data
    map_center = [df_clean['LAT'].mean(), df_clean['LON'].mean()]
    crime_map = folium.Map(location=map_center, zoom_start=12)
    HeatMap(df_clean[['LAT', 'LON']].values.tolist()).add_to(crime_map)
    crime_map.save('crime_map.html')
    print("Heat map created and saved as crime_map.html")



## Interactive Heat Map Display 
Provides seamless visualization display using IPython's IFrame widget to showcase geographic crime patterns without leaving the notebook environment.


In [ ]:
from IPython.display import IFrame
IFrame('crime_map.html', width=700, height=500)


## Data Preprocessing
Prepares the crime dataset for machine learning by removing irrelevant, redundant, and high-missing-value columns that don't contribute to predictive modeling.
Ensures data quality by eliminating features that could cause overfitting, data leakage, or poor model performance due to sparsity.


In [ ]:
# Drop columns that won't help in prediction
csv_path = files_dir = os.path.join(os.environ['KAGGLEHUB_CACHE'], "datasets/sonawanelalitsunil/crime-trends-2020present/versions/1/Crime_Data_from_2020_to_Present.csv")
if not os.path.exists(csv_path):
    print(f"!!!!! Path ({DIR_PATH} not found -- Must fix")
df = pd.read_csv(csv_path)  
# Keep useful features, engineer better ones
# features_to_engineer = [
#     'AREA',           # Police district - good predictor
#     'AREA NAME',      # District name - categorical
#     'Crm Cd',         # Primary crime code - essential
#     'Crm Cd Desc',    # Crime description - useful
#     'Vict Age',       # Victim demographics - relevant
#     'Vict Sex',       # Gender - some missing but useful
#     'Vict Descent',   # Ethnicity - some missing but relevant
#     'Premis Cd',      # Premise type - location context
#     'Premis Desc',    # Premise description - useful
#     'Status Desc'     # Case status - outcome related
# ]

cols_to_drop = [
    'DR_NO',           # Unique identifier - no predictive value
    'Date Rptd',       # Date reported - potential data leakage
    'DATE OCC',        # Date occurred - potential data leakage  
    'TIME OCC',        # Time occurred - too granular
    'LOCATION',        # Text address - redundant with LAT/LON
    'Cross Street',    # missing values
    'LAT',             # Geographic coordinates - may cause overfitting
    'LON',             # Geographic coordinates - may cause overfitting
    'Weapon Used Cd',  # missing values
    'Weapon Desc',     # missing values
    'Crm Cd 2',        # missing values
    'Crm Cd 3',        # missing values
    'Crm Cd 4'         # missing values
]

df = df.drop(columns=cols_to_drop)

## Missing Data Handling 
Cleans the dataset by removing records with missing target values and filling remaining missing data with placeholder values.
Ensures data completeness for machine learning algorithms that cannot handle missing values during training and prediction.


In [ ]:
# Drop rows with missing target
df = df.dropna(subset=['Crm Cd Desc'])

# Fill missing values with a placeholder
df.fillna("Unknown", inplace=True)

## Categorical Data Encoding 
Converts all categorical and float columns to numerical format using Label Encoding, making the data compatible with machine learning algorithms.
Stores all label encoders for potential future use in decoding predictions or transforming new data with the same encoding scheme.


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

# Convert all values to string type first to avoid mixed types
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype == 'float64':
        df[col] = df[col].astype(str)  # Convert entire column to string
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

## Feature and Target Variable Separation
Separates the dataset into feature matrix (X) and target vector (y) for supervised machine learning training.
Prepares the encoded data for model training by isolating predictor variables from the crime type classification target. In short, we're predicting Crime Types based on incident characteristics.


In [ ]:
X = df.drop(columns=['Crm Cd Desc'])     # Features
y = df['Crm Cd Desc']                    # Target (already label-encoded)

## Train-Test Data Split for Model Validation
Divides the preprocessed dataset into training and testing sets using an 80/20 split for proper model evaluation and validation.
Ensures unbiased performance assessment by reserving 20% of data for testing while maintaining reproducible results with a fixed random seed.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Multi-Algorithm Model Training and Evaluation
Trains and compares multiple machine learning models (Decision Tree, Random Forest, Logistic Regression) on the crime prediction task with optimized hyperparameters and proper feature scaling.
Evaluates each model's performance on the test set and provides accuracy comparisons to identify the best-performing algorithm for crime type classification.

`Note`: Logistic Regression will take approximately **25 minutes** to run. Please be patient while the model trains.
 


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Initialize the results dictionary
results = {}

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Updated models with regularization
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_split=50, random_state=42),
    'Random Forest': RandomForestClassifier(max_depth=15, n_estimators=100, n_jobs=-1, random_state=42),
    'Logistic Regression': LogisticRegression(solver='saga', max_iter=100,  n_jobs=-1, random_state=42) # 30 mins

}

# Model evaluation loop
for name, model in models.items():
    try:
        if name == 'Logistic Regression':
            model.fit(X_train_scaled, y_train)
            acc = model.score(X_test_scaled, y_test)
        else:
            model.fit(X_train, y_train)
            acc = model.score(X_test, y_test)
        
        results[name] = acc * 100
        print(f"{name} Accuracy: {acc*100:.2f}%")
    except Exception as e:
        print(f"{name} failed: {e}")

# Display results
print("\nFinal Results:")
for model_name, accuracy in results.items():
    print(f"{model_name}: {accuracy:.2f}%")

